# IR Vibrational Spectroscopy Widget

This notebook demonstrates the IR vibrational widget for displaying quantum chemistry calculation results.

The widget uses **cclib** to parse various quantum chemistry output formats and displays:
- A sortable table of vibrational frequencies and intensities
- An interactive IR spectrum plot with optional broadening

In [1]:
# File handling
from pathlib import Path

# Data manipulation
import numpy as np

# Visualizations
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# %matplotlib inline
# plt.style.use('seaborn-v0_8-notebook')
# mpl.rcParams['axes.spines.right'] = False
# mpl.rcParams['axes.spines.top'] = False

In [2]:
from ir_widget import IRWidget
import numpy as np

## Example 1: Water Molecule (H₂O)

Water has 3 normal modes:
- Symmetric stretch (~3657 cm⁻¹)
- Bending mode (~1595 cm⁻¹)
- Asymmetric stretch (~3756 cm⁻¹)

In [3]:
# Water molecule IR data
frequencies = np.array([1595.0, 3657.0, 3756.0])
intensities = np.array([75.0, 20.0, 45.0])

widget_h2o = IRWidget()
widget_h2o.load_data(frequencies, intensities, formula="H2O")
widget_h2o

## Example 2: Benzene (C₆H₆)

A more complex molecule with multiple vibrational modes across the IR spectrum.

In [4]:
# Simulate benzene-like IR spectrum
np.random.seed(42)

frequencies_benzene = np.concatenate([
    np.random.uniform(400, 800, 8),    # Low frequency modes
    np.random.uniform(900, 1600, 12),  # Mid frequency modes
    np.random.uniform(2800, 3100, 10), # CH stretch modes
])
frequencies_benzene = np.sort(frequencies_benzene)
intensities_benzene = np.random.exponential(30, 30)

widget_benzene = IRWidget()
widget_benzene.load_data(frequencies_benzene, intensities_benzene, formula="C6H6")
widget_benzene

## Customizing the Display

You can adjust various parameters to customize how the spectrum is displayed:

In [5]:
# Change broadening type
widget_benzene.broadening = "gaussian"  # Options: "none", "lorentzian", "gaussian"

# Adjust peak width
widget_benzene.fwhm = 25.0  # Full-width at half-maximum in cm⁻¹

# Adjust spectrum range
widget_benzene.x_min = 500
widget_benzene.x_max = 3500

# Toggle table/plot display
widget_benzene.show_table = True
widget_benzene.show_plot = True

## Loading from a Calculation File

To load data from an actual quantum chemistry calculation:

In [6]:
# Psi4 library
import psi4
for x in Path().glob('psi.*.clean'): x.unlink() 

psi4.core.be_quiet()

In [7]:
# H3O+, planar geometry calcs

# For this calculation, we want to save the output file for later retrieval
# Set a folder name and file name.
outfolder = Path('sample_data')
outfolder.mkdir(exist_ok=True) # Create a new directory, no error if it already exists
outfile = Path(outfolder, 'planar_h3o.log')# fill in a filename
# Remove old file if it exists 
outfile.unlink(missing_ok=True)
# Make sure Psi4 prints the outfile and all required components are present
psi4.set_output_file(outfile, append=True, print_header=True)

# Now we set up the molecule
charge = 1
multp = 1

method = 'hf'
basis = '6-31g(d,p)'

# Read in the contents of our geometry file
planar_geom = """
O	0.00	0.00	0.00
H	0.92	-0.53	0.00
H	-0.92	-0.52	0.00
H	0.00	1.06	0.00
"""
# Again, use f-strings to fill in values for charge, multp, geometry, etc.
planar_h3o = psi4.geometry(f"""
{charge} {multp}

{planar_geom}

""")

## Set options and run the geometry optimization
# This lets us avoid calling the basis for each calculation step.
# Must set reference to 'uhf' or 'rhf' to get IR intensities.
psi4.set_options({
    'reference': 'rhf',
    'basis': basis})

psi4.optimize(method)

# Now calculate the vibrational frequencies for the system
psi4.frequency(method)

# Close the outfile with success line 
if type(psi4.variable('current energy')) is float:
    psi4.extras.exit_printing(success=True)
    psi4.core.close_outfile()

Optimizer: Optimization complete!


In [8]:
# Load from a Gaussian, ORCA, Psi4, or other supported output file
widget = IRWidget(file_path="sample_data/planar_h3o.log")
widget

# Or load after creation
# widget = IRWidget()
# widget.load_file("path/to/your/calculation.log")

'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte


In [9]:
import cclib

In [10]:
planar_log = cclib.io.ccread("sample_data/planar_h30.log")

for key, value in planar_log.metadata.items():
    print(f'{key:18}: {value}')

'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte
'utf-8' codec can't decode byte 0xb6 in position 0: invalid start byte


UnboundLocalError: cannot access local variable 'constructed_data' where it is not associated with a value

In [11]:
cclib.__version__

'2.0a1.post243+d28602f8'

## Accessing the Data

You can access the parsed vibrational data programmatically:

In [ ]:
# Get the data dictionary
data = widget_h2o.data

print(f"Molecular formula: {data['formula']}")
print(f"Number of modes: {data['n_modes']}")
print("\nVibrational modes:")

for mode in data['modes']:
    print(f"  Mode {mode['mode']:2d}: {mode['frequency']:7.2f} cm⁻¹  "
          f"Intensity: {mode['intensity']:7.4f} km/mol")

## Supported File Formats

The widget uses cclib to parse quantum chemistry output files. Supported formats include:

- **Gaussian** (.log, .out, .fchk)
- **ORCA** (.out)
- **Psi4** (.out)
- **NWChem** (.out)
- **GAMESS** (.log, .out)
- **Q-Chem** (.out)
- **Molpro** (.out)
- **And many more!**

The file format is automatically detected by cclib.